## Download and clean metadata.csv

In [1]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id="iulik-pisik/ro_vsr", 
    repo_type="dataset",
    local_dir="./ro_vsr_dataset",
    ignore_patterns=["data/*"]
)

print(f"Files downloaded to: {local_path}")

Fetching 62 files:   0%|          | 0/62 [00:00<?, ?it/s]

Files downloaded to: /home/radumicea/Desktop/Lexical/ro_vsr/ro_vsr_dataset


In [2]:
import pandas as pd

df = pd.read_csv('ro_vsr_dataset/metadata.csv')

df.head(50)

/tmp/ipykernel_17413/1182430212.py:3: DtypeWarning: Columns (2,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('ro_vsr_dataset/metadata.csv')


,file_path,transcript_whisper,transcript_human,annotation_category,is_verified,is_valid_video,split,duration,gender
0,eugenia_serban_y7j5T-AHxTc_clip_1,am fost întotdeauna bine atunci a fost când am...,am fost întotdeauna bine atunci a fost când am...,corect,True,True,test,9.04,F
1,eugenia_serban_y7j5T-AHxTc_clip_2,și da mă consideră un vingătoare,și da mă consider o învingătoare,corect,True,True,test,4.36,F
2,eugenia_serban_y7j5T-AHxTc_clip_3,primul lucru pe care îl fac de cele mai multe...,primul lucru pe care îl fac de cele mai multe ...,corect,True,True,test,15.16,F
3,eugenia_serban_y7j5T-AHxTc_clip_4,dar încet încet am reînvățat drumul către,dar încet încet am reînvățat drumul către,corect,True,True,test,5.12,F
4,eugenia_serban_y7j5T-AHxTc_clip_6,și după care m fac cafeaua știi că sunt o fumă...,și după care îmi fac cafeaua știi că sunt o fu...,audio tăiat,True,False,test,5.52,F
5,eugenia_serban_y7j5T-AHxTc_clip_7,și poro mince via ziua depindă uneori am ședi...,și pe urmă îmi încep iar ziua depinde uneori a...,corect,True,True,test,5.88,F
6,eugenia_serban_y7j5T-AHxTc_clip_8,intrat de câteva luni și pe care îl dezvoltăm,intrat de câteva luni și pe care îl dezvoltăm,corect,True,True,test,4.48,F
7,eugenia_serban_y7j5T-AHxTc_clip_9,deori sunt în studio pentru că mai fac și audi...,deori sunt în studio pentru că mai fac și audi...,audio tăiat,True,False,test,4.88,F
8,eugenia_serban_y7j5T-AHxTc_clip_10,teatru la teatru național sper curând să reușe...,teatru la teatru național sper curând să reușe...,corect,True,True,test,9.00,F
9,eugenia_serban_y7j5T-AHxTc_clip_11,prietenii copilul cu toate supărările pe care ...,prietenii copilul cu toate supărările pe care ...,mai multe voci,True,False,test,8.56,F


In [3]:
df.loc[~df['is_verified']].shape[0]

13890

In [4]:
df = df.loc[df['is_verified']]
df.shape[0]

56585

In [5]:
valid_mask = df['is_valid_video'].astype(str).str.lower().isin(['true'])
invalid_count = (~valid_mask).sum()
int(invalid_count)

16402

In [6]:
df = df.loc[valid_mask].copy()
df.shape[0]

40183

In [7]:
float(df['duration'].sum() / 60 / 60)

127.08757777777778

In [8]:
df['split'].unique()

array(['test', 'trainval', 'test_seen', 'test_beard', 'test_occluded'],
      dtype=object)

In [9]:
float(df.loc[df['split'] == 'trainval']['duration'].sum() / 60 / 60)

116.58713333333336

In [10]:
float(df.loc[df['split'] != 'trainval']['duration'].sum() / 60 / 60)

10.500444444444446

In [11]:
test = df.loc[df['split'] != 'trainval'].copy()
test['split'] = 'test'
print(float(test['duration'].sum() / 60 / 60))

trainval = df.loc[df['split'] == 'trainval'].copy()
train = trainval.sample(frac=0.9, random_state=42).copy()
train['split'] = 'train'
print(float(train['duration'].sum() / 60 / 60))

val = trainval.drop(train.index).copy()
val['split'] = 'val'
print(float(val['duration'].sum() / 60 / 60))

10.500444444444446
104.85466666666666
11.732466666666669


In [12]:
df = pd.concat([train, val, test], ignore_index=True)

assert (df['file_path'].notna() & (df['file_path'].str.strip() != '')).all(), 'Found missing file paths in cleaned dataset'
assert (df['transcript_human'].notna() & (df['transcript_human'].str.strip() != '')).all(), 'Found missing transcripts in cleaned dataset'
assert set(df['split'].unique()) == {'train', 'val', 'test'}, 'Unexpected split labels in cleaned dataset'
assert (df['duration'].notna() & (df['duration'] > 0)).all(), 'Found missing durations in cleaned dataset'

df.to_csv('metadata_clean.csv', index=False)

## Preprocess data

In [13]:
import pandas as pd

df = pd.read_csv('metadata_clean.csv')

df.head(50)

,file_path,transcript_whisper,transcript_human,annotation_category,is_verified,is_valid_video,split,duration,gender
0,andreea_antonescu_qahn56pteSA_clip_238,că au avut ceva de învățat de aici și sigur da...,că au avut ceva de învățat de aici și sigur da...,corect,True,True,train,10.56,F
1,bianca_nutu_4QMZpa_zYKU_clip_81,ceea ce gândești ceea ce simți despre ceea ce ...,ceea ce gândești ceea ce simți despre ceea ce ...,corect,True,True,train,25.76,F
2,iulia_parlea_AMpklgShYD0_clip_279,mai că mai avem o chestie cu mințitul deci ea ...,maică-mea avea o chestie cu mințitul deci ea d...,corect,True,True,train,8.92,F
3,ana_morodan_aDOzKnPltgw_clip_210,tinerilor care au probleme din zona digitală,tinerilor care au probleme din zona digitală,corect,True,True,train,4.28,F
4,lolrelai_sCOjyP2ZdJo_clip_400,doi la mână eu am aflat că nu s-au luat camere...,doi la mână eu am aflat că nu s-au luat camere...,corect,True,True,train,16.28,F
5,irina_petrea_CO4P03D_Hl0_clip_190,dar fără să fiu foarte insistent dacă refuză s...,dar fără să fiu foarte insistent dacă refuză s...,corect,True,True,train,4.40,F
6,ioana_ginghina_WC5SA66nEIY_clip_68,cei doi clunii și cu brad pitt frumoși așa și...,cei doi clooney și cu brad pitt frumoși așa ș...,corect,True,True,train,7.56,F
7,mihaela_tatu_kD4lJehL23Q_clip_291,nu nu seara între culcare îi mulțumesc pentru ...,nu nu seara înainte de culcare îi mulțumesc pe...,corect,True,True,train,15.76,F
8,adrian_alexandrov_G1h5PBrtzds_clip_85,nu pe mine personal nu toți prietenii mei sun...,nu pe mine personal nu toți prietenii mei sunt...,corect,True,True,train,19.64,M
9,catalin_bordea_zlRnts0-FWg_clip_179,mă știu de foarte mult timp cu ei dar acuma no...,mă știu de foarte mult timp cu ei dar acum noi...,corect,True,True,train,15.48,M


In [14]:
df = df[['file_path', 'transcript_human', 'split']].rename(columns={
    'file_path': 'clip_path',
    'transcript_human': 'transcript',
})

In [15]:
# Check non alpha characters in transcripts

bad_rows = df[df['transcript'].str.contains(r'[^a-zA-ZĂăÂâÎîȘșȚț\-\s]', regex=True)]
print(f"Found {len(bad_rows)} rows with non-alphabetic characters in transcripts:")

for text in bad_rows['transcript'].head().values:
    print(text)

Found 254 rows with non-alphabetic characters in transcripts:
m-am logodit și mi-a fugit de la deci știi cum ne-am despărțit nu vreau să că s-a și măritat acum are copil dar vreau să zic că filmam o videoclipă am fi luat drepturile pentru je t'aime de la lara
tocmai ce am schimbat setul motor i-am pus pompă de injecție nouă i-am făcut nu știu ce zice dar tre' s-o vând că se-mplinesc nu știu câți ani și noi cu uber-ul n-avem voie să ținem mașina mai mulți ani nu știu ce
s-a cam spart echipa ca să zic așa și am rămas îți dai seama acu' când m-am reîntors
da' dacă nu îmi concentrez atenția să fie pe mai multe locuri nu
este o chestie normală tataie vrei să mănânci nu vreau lăsați-mă în pace punct asta era discuția noastră știi era și foarte bătrân într-adevăr și cumva ne obișnuisem cu prezența lui că e acolo în cameră tot timpul și deodată s-a stins și abia atunci am conștientizat bă da' noi n-am făcut nimic frate să îl ajutăm și noi să iasă din casă să ne mai ducem peste el să-i cântăm


In [16]:
df.drop(bad_rows.index, inplace=True)
df.to_csv('metadata_ready.csv', index=False)

## Extract clips

In [ ]:
from pathlib import Path
import shutil
import tarfile

# Project root is one level above this notebook folder.
project_root = Path.cwd().resolve().parent
data_dir = project_root / "ro_vsr" / "ro_vsr_dataset" / "data"
clips_dir = project_root / "clips"

if not data_dir.exists():
    raise FileNotFoundError(f"Data folder not found: {data_dir}")

tar_files = sorted(data_dir.glob("*.tar"))
if not tar_files:
    raise FileNotFoundError(f"No .tar files found in: {data_dir}")

clips_dir.mkdir(parents=True, exist_ok=True)

# Extract each tar into clips/<tar_name_without_extension>/.
for tar_path in tar_files:
    target_dir = clips_dir / tar_path.stem
    target_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, mode="r") as tf:
        tf.extractall(path=target_dir)

    # Flatten duplicated nested layout: target_dir/target_dir_name/* -> target_dir/*
    nested_same_name = target_dir / target_dir.name
    if nested_same_name.is_dir():
        for item in nested_same_name.iterdir():
            destination = target_dir / item.name
            if destination.exists():
                raise FileExistsError(
                    f"Cannot flatten {nested_same_name}: destination already exists -> {destination}"
                )
            shutil.move(str(item), str(destination))
        nested_same_name.rmdir()

# Validate extraction before deleting source data.
failed = []
for tar_path in tar_files:
    target_dir = clips_dir / tar_path.stem

    if not target_dir.exists() or not target_dir.is_dir():
        failed.append((tar_path.name, "target folder missing"))
        continue

    # Ensure there are files directly in target_dir (no extra duplicated level).
    top_level_files = [p for p in target_dir.iterdir() if p.is_file()]
    if not top_level_files:
        failed.append((tar_path.name, "no files found directly under target folder"))
        continue

    avi_files = [p for p in top_level_files if p.suffix.lower() == ".avi"]
    if not avi_files:
        failed.append((tar_path.name, "no .avi files found directly under target folder"))

if failed:
    print("Extraction check failed. data was NOT deleted.")
    for name, reason in failed:
        print(f" - {name}: {reason}")
else:
    shutil.rmtree(data_dir)
    print("All archives extracted and flattened successfully.")
    print(f"Extracted clips are in: {clips_dir}")
    print(f"Deleted original data folder: {data_dir}")